# Data Collection: ETH/ETC Historical Data

This notebook demonstrates how to collect historical price data for ETH and ETC from Bitvavo.

## Goals
1. Fetch historical OHLCV data for ETH-EUR and ETC-EUR
2. Explore the data structure
3. Visualize price movements
4. Save data for future analysis

In [ ]:
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root))

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from statistical_arbitrage.data.bitvavo_client import BitvavoDataCollector, fetch_eth_etc_data

## 1. Initialize Bitvavo Client

In [ ]:
# Initialize collector (no API keys needed for public data)
collector = BitvavoDataCollector()

## 2. Check Available Markets

In [ ]:
# Get all available markets
markets_df = collector.get_available_markets()
print(f"Total markets available: {len(markets_df)}")

# Filter for EUR pairs
eur_markets = markets_df.filter(pl.col("market").str.ends_with("-EUR"))
print(f"\nEUR markets: {len(eur_markets)}")

# Check if ETH-EUR and ETC-EUR are available
target_pairs = ["ETH-EUR", "ETC-EUR"]
available_targets = eur_markets.filter(pl.col("market").is_in(target_pairs))
print(f"\nTarget pairs available:")
print(available_targets.select(["market", "status"]))

## 3. Fetch Historical Data

Let's fetch 90 days of hourly data for both ETH and ETC.

In [ ]:
# Fetch data using convenience function
eth_df, etc_df = fetch_eth_etc_data(
    interval="1h",
    days_back=90,
    save=True
)

In [ ]:
# Inspect ETH data
print("ETH-EUR Data:")
print(f"Shape: {eth_df.shape}")
print(f"Date range: {eth_df['datetime'].min()} to {eth_df['datetime'].max()}")
print("\nFirst few rows:")
print(eth_df.head())
print("\nData types:")
print(eth_df.dtypes)

In [ ]:
# Inspect ETC data
print("ETC-EUR Data:")
print(f"Shape: {etc_df.shape}")
print(f"Date range: {etc_df['datetime'].min()} to {etc_df['datetime'].max()}")
print("\nFirst few rows:")
print(etc_df.head())

## 4. Basic Statistics

In [ ]:
# Summary statistics for ETH
print("ETH-EUR Summary Statistics:")
print(eth_df.select(["open", "high", "low", "close", "volume"]).describe())

In [ ]:
# Summary statistics for ETC
print("ETC-EUR Summary Statistics:")
print(etc_df.select(["open", "high", "low", "close", "volume"]).describe())

## 5. Visualize Price Data

In [ ]:
# Create subplots for ETH and ETC prices
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("ETH-EUR Price", "ETC-EUR Price"),
    vertical_spacing=0.1,
    shared_xaxes=True,
)

# ETH candlestick
fig.add_trace(
    go.Candlestick(
        x=eth_df["datetime"].to_list(),
        open=eth_df["open"].to_list(),
        high=eth_df["high"].to_list(),
        low=eth_df["low"].to_list(),
        close=eth_df["close"].to_list(),
        name="ETH"
    ),
    row=1, col=1
)

# ETC candlestick
fig.add_trace(
    go.Candlestick(
        x=etc_df["datetime"].to_list(),
        open=etc_df["open"].to_list(),
        high=etc_df["high"].to_list(),
        low=etc_df["low"].to_list(),
        close=etc_df["close"].to_list(),
        name="ETC"
    ),
    row=2, col=1
)

fig.update_layout(
    title="ETH and ETC Historical Prices (90 days)",
    height=800,
    showlegend=False,
    xaxis_rangeslider_visible=False,
    xaxis2_rangeslider_visible=False,
)

fig.update_yaxes(title_text="Price (EUR)", row=1, col=1)
fig.update_yaxes(title_text="Price (EUR)", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.show()

## 6. Compare Normalized Prices

Normalize both series to see relative movements.

In [ ]:
# Normalize prices (set first value to 100)
eth_normalized = (eth_df["close"] / eth_df["close"][0] * 100).to_list()
etc_normalized = (etc_df["close"] / etc_df["close"][0] * 100).to_list()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=eth_df["datetime"].to_list(),
    y=eth_normalized,
    name="ETH-EUR",
    line=dict(color="blue", width=2)
))

fig.add_trace(go.Scatter(
    x=etc_df["datetime"].to_list(),
    y=etc_normalized,
    name="ETC-EUR",
    line=dict(color="orange", width=2)
))

fig.update_layout(
    title="ETH vs ETC - Normalized Price Comparison (Base = 100)",
    xaxis_title="Date",
    yaxis_title="Normalized Price",
    height=600,
    hovermode="x unified"
)

fig.show()

## 7. Quick Correlation Check

In [ ]:
# Merge dataframes on timestamp for correlation analysis
merged_df = eth_df.select(["timestamp", pl.col("close").alias("eth_close")]).join(
    etc_df.select(["timestamp", pl.col("close").alias("etc_close")]),
    on="timestamp",
    how="inner"
)

# Calculate correlation
correlation = merged_df.select(
    pl.corr("eth_close", "etc_close").alias("correlation")
)["correlation"][0]

print(f"\nPearson Correlation between ETH and ETC: {correlation:.4f}")
print("\nNote: Correlation measures linear relationship, but for pairs trading,")
print("we need to check cointegration (next notebook).")

## Summary

In this notebook, we:
1. Connected to Bitvavo API
2. Fetched 90 days of hourly data for ETH-EUR and ETC-EUR
3. Explored basic statistics
4. Visualized price movements
5. Checked basic correlation

## Next Steps

In the next notebook, we'll:
- Test for cointegration between ETH and ETC
- Calculate the spread
- Analyze mean-reversion properties
- Determine if this pair is suitable for statistical arbitrage